# Thomann Order Analysis

Exploratory analysis of my Thomann.de order history.

In [ ]:
import json
import glob
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 110

# Load the most recent export file
files = sorted(glob.glob("orders-*.json"))
with open(files[-1]) as f:
    data = json.load(f)

# Build DataFrames
orders = pd.DataFrame(data["orders"])
orders["date"] = pd.to_datetime(orders["dateIso"])
orders["year"] = orders["date"].dt.year

items = pd.json_normalize(data["orders"], record_path="items", meta=["orderNumber", "dateIso"])
items["date"] = pd.to_datetime(items["dateIso"])
items["year"] = items["date"].dt.year

print(f"Loaded {len(orders)} orders with {len(items)} items from {files[-1]}")
print(f"Date range: {orders['date'].min():%Y-%m-%d} to {orders['date'].max():%Y-%m-%d}")

## Yearly Summary

Total spending and number of orders per year.

In [ ]:
yearly = orders.groupby("year").agg(
    total_spent=("totalSumNum", "sum"),
    num_orders=("orderNumber", "count"),
    avg_order=("totalSumNum", "mean"),
).rename_axis("Year")

yearly["total_spent"] = yearly["total_spent"].round(2)
yearly["avg_order"] = yearly["avg_order"].round(2)
yearly.columns = ["Total Spent (€)", "Number of Orders", "Avg Order (€)"]

# Add a totals row
totals = pd.DataFrame({
    "Total Spent (€)": [yearly["Total Spent (€)"].sum()],
    "Number of Orders": [yearly["Number of Orders"].sum()],
    "Avg Order (€)": [yearly["Total Spent (€)"].sum() / yearly["Number of Orders"].sum()],
}, index=["TOTAL"])
totals["Avg Order (€)"] = totals["Avg Order (€)"].round(2)

pd.concat([yearly, totals])

## Spending Over Time

In [ ]:
fig, ax1 = plt.subplots()

color_spend = "#2563eb"
color_count = "#f59e0b"

ax1.bar(yearly.index, yearly["Total Spent (€)"], color=color_spend, alpha=0.8, label="Total Spent")
ax1.set_xlabel("Year")
ax1.set_ylabel("Total Spent (€)", color=color_spend)
ax1.tick_params(axis="y", labelcolor=color_spend)
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f} €"))

ax2 = ax1.twinx()
ax2.plot(yearly.index, yearly["Number of Orders"], color=color_count, marker="o", linewidth=2, label="# Orders")
ax2.set_ylabel("Number of Orders", color=color_count)
ax2.tick_params(axis="y", labelcolor=color_count)

fig.suptitle("Yearly Thomann Spending", fontsize=14, fontweight="bold")
fig.tight_layout()
plt.show()

## Cumulative Spending

In [ ]:
cumulative = orders.sort_values("date").set_index("date")["totalSumNum"].cumsum()

fig, ax = plt.subplots()
ax.fill_between(cumulative.index, cumulative.values, alpha=0.3, color="#2563eb")
ax.plot(cumulative.index, cumulative.values, color="#2563eb", linewidth=2)
ax.set_ylabel("Cumulative Spending (€)")
ax.set_xlabel("")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f} €"))
ax.set_title("Cumulative Thomann Spending Over Time", fontsize=14, fontweight="bold")
fig.tight_layout()
plt.show()

## Orders by Month (Seasonal Pattern)

In [ ]:
orders["month"] = orders["date"].dt.month
heatmap_data = orders.groupby(["year", "month"])["totalSumNum"].sum().unstack(fill_value=0)

fig, ax = plt.subplots(figsize=(12, 6))
im = ax.imshow(heatmap_data.values, aspect="auto", cmap="YlOrRd")
ax.set_xticks(range(12))
ax.set_xticklabels(["Jan", "Feb", "Mar", "Apr", "May", "Jun",
                     "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"])
ax.set_yticks(range(len(heatmap_data.index)))
ax.set_yticklabels(heatmap_data.index)
ax.set_title("Monthly Spending Heatmap (€)", fontsize=14, fontweight="bold")

cbar = fig.colorbar(im, ax=ax, format=mticker.FuncFormatter(lambda x, _: f"{x:,.0f} €"))
fig.tight_layout()
plt.show()

## Top 10 Most Expensive Items Per Year

In [ ]:
for year in sorted(items["year"].unique()):
    year_items = items[items["year"] == year].nlargest(10, "lineTotalNum")
    top = year_items[["name", "lineTotalNum", "count"]].copy()
    top.columns = ["Item", "Total (€)", "Qty"]
    top = top.reset_index(drop=True)
    top.index = top.index + 1
    print(f"\n{'='*60}")
    print(f"  {year}")
    print(f"{'='*60}")
    print(top.to_string())